# BirdCLEF 2026 — Inference & Submission Generation

In [ ]:
import sys
sys.path.append('..')

import torch
import pandas as pd
from pathlib import Path

from src.config import load_config, ROOT
from src.models import BirdCLEFModel
from src.inference import generate_submission

In [ ]:
config = load_config('../configs/baseline.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_csv(ROOT / config['train_csv'])
species_list = sorted(df['primary_label'].unique().tolist())

model = BirdCLEFModel(num_classes=len(species_list), model_name=config['model_name'], pretrained=False)
checkpoint = ROOT / config['output_dir'] / 'best_model.pth'
model.load_state_dict(torch.load(checkpoint, map_location=device))
model = model.to(device)
model.eval()
print('Model loaded.')

In [ ]:
test_dir = ROOT / config['test_soundscapes_dir']
test_files = sorted(test_dir.glob('*.ogg'))
print(f'Found {len(test_files)} test soundscapes')

output_path = ROOT / config['submission_path']
output_path.parent.mkdir(parents=True, exist_ok=True)

generate_submission(model, [str(f) for f in test_files], species_list, config, device, output_path)
print('Done!')

In [ ]:
submission = pd.read_csv(output_path)
print(f'Submission shape: {submission.shape}')
submission.head()